In [1]:
from pathlib import Path

import torch
import pandas as pd
import numpy as np
import json
import hashlib
import transformers
import peft

from datasets import load_from_disk

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    set_seed
)

from peft import PeftModel

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data"
models_dir = project_dir / "models"
reports_dir = project_dir / "reports"
configs_dir = project_dir / "configs"

configs_dir.mkdir(parents=True, exist_ok=True)

SEED = 42

set_seed(SEED)

assert torch.cuda.is_available()
assert torch.cuda.is_bf16_supported()

device = "cuda"

print("Project Directory:", project_dir)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 Support:", torch.cuda.is_bf16_supported())
print("Transformers Version:", transformers.__version__)
print("PEFT Version:", peft.__version__)

/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project Directory: /home/pasali/Desktop/Data/AI-Engineer/Hafta-7/HealthBrief
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
BF16 Support: True
Transformers Version: 5.14.1
PEFT Version: 0.22.0


In [2]:
BASE_MODEL = "google/flan-t5-small"

BASE_REVISION = "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab"

ADAPTER_PATH = models_dir / "healthbrief_best_adapter"

CHECKPOINT = "checkpoint-96"

SOURCE_SELECTION_VERSION = "V1"

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 768

GENERATION_MAX_LENGTH = 256
GENERATION_NUM_BEAMS = 1

REPETITION_PENALTY = 1.0
NO_REPEAT_NGRAM_SIZE = 3

DO_SAMPLE = False
USE_BF16 = True

assert ADAPTER_PATH.exists()

print("Base Model:", BASE_MODEL)
print("Base Revision:", BASE_REVISION)
print("Adapter Path:", ADAPTER_PATH)
print("Checkpoint:", CHECKPOINT)
print("Source Selection:", SOURCE_SELECTION_VERSION)

Base Model: google/flan-t5-small
Base Revision: 0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab
Adapter Path: /home/pasali/Desktop/Data/AI-Engineer/Hafta-7/HealthBrief/models/healthbrief_best_adapter
Checkpoint: checkpoint-96
Source Selection: V1


In [4]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_REVISION
)

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)

loaded_base_revision = getattr(
    base_model.config,
    "_commit_hash",
    None
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False
)

model.to(device)

model.eval()

model.print_trainable_parameters()

print("Model Type:", type(model).__name__)
print("Model Device:", next(model.parameters()).device)
print("Model Dtype:", next(model.parameters()).dtype)
print("Training Mode:", model.training)
print("Loaded Base Revision:", loaded_base_revision)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3592.86it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 0 || all params: 77,305,216 || trainable%: 0.0000
Model Type: PeftModelForSeq2SeqLM
Model Device: cuda:0
Model Dtype: torch.float32
Training Mode: False
Loaded Base Revision: 0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab


In [5]:
model.generation_config.max_length = GENERATION_MAX_LENGTH
model.generation_config.max_new_tokens = None
model.generation_config.num_beams = GENERATION_NUM_BEAMS
model.generation_config.repetition_penalty = REPETITION_PENALTY
model.generation_config.no_repeat_ngram_size = NO_REPEAT_NGRAM_SIZE
model.generation_config.do_sample = DO_SAMPLE

total_parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameter_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_parameter_count = (
    total_parameter_count - trainable_parameter_count
)

assert total_parameter_count == 77305216
assert trainable_parameter_count == 0
assert not model.training

if loaded_base_revision is not None:
    assert loaded_base_revision == BASE_REVISION

print(model.generation_config)

print("Total Parameters:", total_parameter_count)
print("Trainable Parameters:", trainable_parameter_count)
print("Frozen Parameters:", frozen_parameter_count)
print("Frozen Model Verification: PASS")

GenerationConfig {
  "decoder_start_token_id": 0,
  "do_sample": false,
  "eos_token_id": 1,
  "max_length": 256,
  "no_repeat_ngram_size": 3,
  "num_beams": 1,
  "pad_token_id": 0,
  "repetition_penalty": 1.0
}

Total Parameters: 77305216
Trainable Parameters: 0
Frozen Parameters: 77305216
Frozen Model Verification: PASS


In [6]:
final_inference_config = {
    "project": "HealthBrief",
    "model": {
        "base_model": BASE_MODEL,
        "base_revision": BASE_REVISION,
        "loaded_base_revision": loaded_base_revision,
        "adapter_path": str(
            ADAPTER_PATH.relative_to(project_dir)
        ),
        "checkpoint": CHECKPOINT,
        "peft_method": "LoRA",
        "lora_rank": 8,
        "total_parameters": total_parameter_count,
        "trainable_parameters": trainable_parameter_count
    },
    "tokenizer": {
        "path": str(
            ADAPTER_PATH.relative_to(project_dir)
        ),
        "class": type(tokenizer).__name__,
        "vocab_size": len(tokenizer),
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id
    },
    "preprocessing": {
        "source_selection_version": SOURCE_SELECTION_VERSION,
        "max_input_length": MAX_INPUT_LENGTH,
        "max_target_length": MAX_TARGET_LENGTH
    },
    "generation": {
        "config_name": "C",
        "max_length": GENERATION_MAX_LENGTH,
        "max_new_tokens": None,
        "num_beams": GENERATION_NUM_BEAMS,
        "repetition_penalty": REPETITION_PENALTY,
        "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
        "do_sample": DO_SAMPLE,
        "forced_eos_token_id": model.generation_config.forced_eos_token_id,
        "use_cache": model.generation_config.use_cache
    },
    "precision": {
        "training_precision": "bf16_mixed_precision",
        "model_dtype": str(next(model.parameters()).dtype),
        "inference_precision": "bf16_autocast",
        "bf16_autocast_enabled": USE_BF16
    },
    "environment": {
        "transformers_version": transformers.__version__,
        "peft_version": peft.__version__,
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0)
    },
    "seed": SEED,
    "evaluation": {
        "model_selection_split": "validation",
        "final_evaluation_split": "final_holdout",
        "configuration_selected_before_holdout": True
    }
}

print(json.dumps(
    final_inference_config,
    indent=4
))

{
    "project": "HealthBrief",
    "model": {
        "base_model": "google/flan-t5-small",
        "base_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
        "loaded_base_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
        "adapter_path": "models/healthbrief_best_adapter",
        "checkpoint": "checkpoint-96",
        "peft_method": "LoRA",
        "lora_rank": 8,
        "total_parameters": 77305216,
        "trainable_parameters": 0
    },
    "tokenizer": {
        "path": "models/healthbrief_best_adapter",
        "class": "T5Tokenizer",
        "vocab_size": 32100,
        "pad_token_id": 0,
        "eos_token_id": 1
    },
    "preprocessing": {
        "source_selection_version": "V1",
        "max_input_length": 512,
        "max_target_length": 768
    },
    "generation": {
        "config_name": "C",
        "max_length": 256,
        "max_new_tokens": null,
        "num_beams": 1,
        "repetition_penalty": 1.0,
        "no_repeat_ngram_size": 3,


In [7]:
final_config_path = configs_dir / "final_inference_config.json"

config_bytes = json.dumps(
    final_inference_config,
    indent=4,
    sort_keys=True,
    ensure_ascii=False
).encode("utf-8")

if final_config_path.exists():

    assert final_config_path.read_bytes() == config_bytes, (
        "Existing frozen configuration differs."
    )

else:

    final_config_path.write_bytes(config_bytes)

print("Configuration Path:", final_config_path)
print("Configuration Exists:", final_config_path.exists())
print("Configuration Size:", final_config_path.stat().st_size)

Configuration Path: /home/pasali/Desktop/Data/AI-Engineer/Hafta-7/HealthBrief/configs/final_inference_config.json
Configuration Exists: True
Configuration Size: 1728


In [8]:
final_config_hash = hashlib.sha256(
    final_config_path.read_bytes()
).hexdigest()

assert len(final_config_hash) == 64

print("Algorithm: SHA-256")
print("Configuration Hash:", final_config_hash)
print("Hash Length:", len(final_config_hash))

Algorithm: SHA-256
Configuration Hash: b2a97efb8e480cdf168288167c9673cbbffd5cfdb4e9e4f03f69d9fb65eaaa39
Hash Length: 64


In [9]:
final_configuration_manifest = pd.DataFrame([{
    "base_model": BASE_MODEL,
    "base_revision": BASE_REVISION,
    "loaded_base_revision": loaded_base_revision,
    "adapter_path": str(
        ADAPTER_PATH.relative_to(project_dir)
    ),
    "checkpoint": CHECKPOINT,
    "source_selection_version": SOURCE_SELECTION_VERSION,
    "max_input_length": MAX_INPUT_LENGTH,
    "generation_config": "C",
    "generation_max_length": GENERATION_MAX_LENGTH,
    "num_beams": GENERATION_NUM_BEAMS,
    "repetition_penalty": REPETITION_PENALTY,
    "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
    "do_sample": DO_SAMPLE,
    "inference_precision": "bf16_autocast",
    "seed": SEED,
    "total_parameters": total_parameter_count,
    "trainable_parameters": trainable_parameter_count,
    "config_path": str(
        final_config_path.relative_to(project_dir)
    ),
    "hash_algorithm": "SHA-256",
    "config_sha256": final_config_hash
}])

manifest_path = reports_dir / "final_configuration_manifest.csv"

if manifest_path.exists():

    existing_manifest = pd.read_csv(manifest_path)

    assert existing_manifest["config_sha256"].iloc[0] == final_config_hash

else:

    final_configuration_manifest.to_csv(
        manifest_path,
        index=False
    )

display(final_configuration_manifest.T)

print("Manifest saved successfully.")

,0
base_model,google/flan-t5-small
base_revision,0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab
loaded_base_revision,0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab
adapter_path,models/healthbrief_best_adapter
checkpoint,checkpoint-96
source_selection_version,V1
max_input_length,512
generation_config,C
generation_max_length,256
num_beams,1


Manifest saved successfully.


In [10]:
from datasets import DatasetDict

sampled_dataset = load_from_disk(
    str(data_dir / "healthbrief_sampled")
)

holdout_dataset = load_from_disk(
    str(data_dir / "final_holdout")
)

if isinstance(holdout_dataset, DatasetDict):
    holdout_dataset = holdout_dataset["final_holdout"] if "final_holdout" in holdout_dataset else holdout_dataset["test"]

manifest_df = pd.read_csv(
    reports_dir / "dataset_split_manifest.csv"
)

with open(
    data_dir / "dataset_metadata.json",
    "r",
    encoding="utf-8"
) as file:
    dataset_metadata = json.load(file)

print("Train Samples:", len(sampled_dataset["train"]))
print("Validation Samples:", len(sampled_dataset["validation"]))
print("Final Holdout Samples:", len(holdout_dataset))
print("Holdout Columns:", holdout_dataset.column_names)
print("Dataset Revision:", dataset_metadata["revision"])

Train Samples: 500
Validation Samples: 100
Final Holdout Samples: 100
Holdout Columns: ['article', 'summary', 'section_headings', 'keywords', 'year', 'title', 'article_id']
Dataset Revision: 144e9c785e3a309da804eb6786feb38e6596f390


In [12]:
def create_article_id(article):

    normalized_article = " ".join(
        article.lower().split()
    )

    return hashlib.sha256(
        normalized_article.encode("utf-8")
    ).hexdigest()


train_article_ids = [
    create_article_id(article)
    for article in sampled_dataset["train"]["article"]
]

validation_article_ids = [
    create_article_id(article)
    for article in sampled_dataset["validation"]["article"]
]

holdout_article_ids = [
    create_article_id(article)
    for article in holdout_dataset["article"]
]

print("Unique Train Articles:", len(set(train_article_ids)))
print("Unique Validation Articles:", len(set(validation_article_ids)))
print("Unique Holdout Articles:", len(set(holdout_article_ids)))

Unique Train Articles: 500
Unique Validation Articles: 100
Unique Holdout Articles: 100


In [14]:
train_ids = set(train_article_ids)

validation_ids = set(validation_article_ids)

holdout_ids = set(holdout_article_ids)

manifest_holdout_ids = set(
    manifest_df.loc[
        manifest_df["split"] == "final_holdout",
        "article_id"
    ]
)

train_holdout_overlap = len(
    train_ids & holdout_ids
)

validation_holdout_overlap = len(
    validation_ids & holdout_ids
)

holdout_duplicate_count = (
    len(holdout_article_ids) - len(holdout_ids)
)

valid_reference_count = sum(
    isinstance(summary, str) and bool(summary.strip())
    for summary in holdout_dataset["summary"]
)

missing_reference_count = (
    len(holdout_dataset) - valid_reference_count
)

manifest_match = (
    holdout_ids == manifest_holdout_ids
)

assert len(holdout_dataset) == 100
assert len(holdout_ids) == 100

assert train_holdout_overlap == 0
assert validation_holdout_overlap == 0

assert holdout_duplicate_count == 0
assert missing_reference_count == 0
assert manifest_match

print("Train-Holdout Overlap:", train_holdout_overlap)
print("Validation-Holdout Overlap:", validation_holdout_overlap)
print("Holdout Duplicates:", holdout_duplicate_count)
print("Valid References:", valid_reference_count)
print("Missing References:", missing_reference_count)
print("Manifest Match:", manifest_match)
print("Holdout Integrity Verification: PASS")

Train-Holdout Overlap: 0
Validation-Holdout Overlap: 0
Holdout Duplicates: 0
Valid References: 100
Missing References: 0
Manifest Match: True
Holdout Integrity Verification: PASS


In [15]:
holdout_integrity_df = pd.DataFrame([{
    "dataset_name": dataset_metadata["dataset"],
    "dataset_revision": dataset_metadata["revision"],
    "seed": SEED,
    "train_samples": len(sampled_dataset["train"]),
    "validation_samples": len(sampled_dataset["validation"]),
    "holdout_samples": len(holdout_dataset),
    "unique_holdout_articles": len(holdout_ids),
    "train_holdout_overlap": train_holdout_overlap,
    "validation_holdout_overlap": validation_holdout_overlap,
    "holdout_duplicate_count": holdout_duplicate_count,
    "valid_reference_count": valid_reference_count,
    "missing_reference_count": missing_reference_count,
    "manifest_match": manifest_match,
    "article_id_algorithm": "normalized_text_sha256",
    "config_sha256": final_config_hash,
    "holdout_used_for_training": False,
    "holdout_used_for_model_selection": False,
    "holdout_used_for_config_selection": False,
    "holdout_evaluation_started": False,
    "integrity_check_pass": (
        len(holdout_dataset) == 100
        and len(holdout_ids) == 100
        and train_holdout_overlap == 0
        and validation_holdout_overlap == 0
        and holdout_duplicate_count == 0
        and missing_reference_count == 0
        and manifest_match
    )
}])

holdout_integrity_df.to_csv(
    reports_dir / "holdout_integrity.csv",
    index=False
)

display(holdout_integrity_df.T)

print("Holdout Integrity Report: Saved Successfully")

,0
dataset_name,BioLaySumm/BioLaySumm2025-eLife
dataset_revision,144e9c785e3a309da804eb6786feb38e6596f390
seed,42
train_samples,500
validation_samples,100
holdout_samples,100
unique_holdout_articles,100
train_holdout_overlap,0
validation_holdout_overlap,0
holdout_duplicate_count,0


Holdout Integrity Report: Saved Successfully


In [20]:
PROMPT = (
    "Summarize the following biomedical research article "
    "in plain language for a general audience:\n\n"
)


def prepare_source_text(article, section_headings, tokenizer, max_length):

    headings = [
        "Abstract", "Introduction", "Results",
        "Discussion", "Conclusion", "Conclusions",
        "Results and Discussion", "Materials and Methods",
        "Methods", "References", "Acknowledgements"
    ]

    if section_headings is not None:
        headings.extend(section_headings)

    headings = {h.strip().lower() for h in headings}

    paragraphs = [
        line.strip()
        for line in article.splitlines()
        if line.strip()
    ]

    sections = {}
    current_section = None
    opening = []

    for paragraph in paragraphs:

        heading, separator, content = paragraph.partition(":")
        heading = heading.strip().lower()

        if heading in headings:

            current_section = heading
            sections.setdefault(heading, [])

            if content.strip():
                sections[heading].append(content.strip())

        elif current_section is None:

            opening.append(paragraph)

        else:

            sections[current_section].append(paragraph)

    sections = {
        name: "\n".join(texts)
        for name, texts in sections.items()
        if texts
    }

    selected = []
    fallback_used = False

    if "abstract" in sections:

        selected.append(("Abstract", sections["abstract"]))

    else:

        opening_text = opening[0] if opening else paragraphs[0]

        selected.append(("Opening (fallback)", opening_text))

        fallback_used = True

    secondary_names = [
        "conclusion",
        "conclusions",
        "discussion",
        "results and discussion",
        "results"
    ]

    secondary_name = next(
        (name for name in secondary_names if name in sections),
        None
    )

    if secondary_name:

        selected.append(
            (secondary_name.title(), sections[secondary_name])
        )

        if secondary_name == "results":
            fallback_used = True

    else:

        fallback_used = True

        keywords = [
            "we found",
            "we show",
            "we demonstrated",
            "we observed",
            "our findings",
            "our results",
            "these results",
            "these findings",
            "taken together",
            "in conclusion"
        ]

        best_paragraph = None
        best_score = 0

        for paragraph in paragraphs[1:]:

            if len(paragraph.split()) < 40:
                continue

            if paragraph in selected[0][1]:
                continue

            score = sum(
                keyword in paragraph.lower()
                for keyword in keywords
            )

            if score > best_score:

                best_score = score
                best_paragraph = paragraph

        if best_paragraph is not None:

            selected.append(
                ("Result-cue paragraph (fallback)", best_paragraph)
            )

    if not sections and len(selected) == 1:

        selected = [("Opening (fallback)", article)]

    def count_tokens(text, special_tokens=False):

        return len(tokenizer.encode(
            text,
            add_special_tokens=special_tokens,
            verbose=False
        ))

    def truncate_text(text, limit):

        if limit <= 0:
            return ""

        offsets = tokenizer(
            text,
            add_special_tokens=False,
            return_offsets_mapping=True,
            verbose=False
        )["offset_mapping"]

        if len(offsets) <= limit:
            return text

        return text[:offsets[limit - 1][1]].strip()

    prompt_tokens = count_tokens(PROMPT, True)

    budget = max_length - prompt_tokens

    if budget <= 0:

        raise ValueError("Insufficient token budget.")

    lengths = [
        count_tokens(text)
        for _, text in selected
    ]

    if len(selected) == 2:

        first = min(lengths[0], budget // 2)

        second = min(lengths[1], budget - first)

        first = min(lengths[0], budget - second)

        limits = [first, second]

    else:

        limits = [min(lengths[0], budget)]

    while True:

        retained = [
            truncate_text(text, limit)
            for (_, text), limit in zip(selected, limits)
        ]

        source_text = "\n\n".join(
            text for text in retained if text
        )

        input_text = PROMPT + source_text

        input_tokens = count_tokens(input_text, True)

        if input_tokens <= max_length:
            break

        limits[limits.index(max(limits))] -= 1

    return {
        "input_text": input_text,
        "source_sections": [
            name for (name, _), text in zip(selected, retained)
            if text
        ],
        "verified_headings": list(sections.keys()),
        "original_token_count": count_tokens(article),
        "retained_token_count": count_tokens(source_text),
        "prompt_token_count": prompt_tokens,
        "input_token_count": input_tokens,
        "truncated": source_text != article.strip(),
        "fallback_used": fallback_used
    }

In [21]:
holdout_prepared_records = []

for sample_id, sample in enumerate(holdout_dataset):

    source_result = prepare_source_text(
        article=sample["article"],
        section_headings=sample["section_headings"],
        tokenizer=tokenizer,
        max_length=MAX_INPUT_LENGTH
    )

    input_text = source_result["input_text"]

    input_tokens = tokenizer(
        input_text,
        truncation=False,
        add_special_tokens=True
    )["input_ids"]

    holdout_prepared_records.append({
        "sample_id": sample_id,
        "article_id": holdout_article_ids[sample_id],
        "input_text": input_text,
        "source_token_count": len(input_tokens),
        "source_sections": source_result["source_sections"],
        "fallback_used": source_result["fallback_used"]
    })

holdout_prepared_df = pd.DataFrame(
    holdout_prepared_records
)

display(holdout_prepared_df[[
    "sample_id",
    "source_token_count",
    "source_sections",
    "fallback_used"
]].head())

print("Prepared Samples:", len(holdout_prepared_df))

,sample_id,source_token_count,source_sections,fallback_used
0,0,512,"[Opening (fallback), Result-cue paragraph (fal...",True
1,1,511,"[Opening (fallback), Result-cue paragraph (fal...",True
2,2,512,"[Opening (fallback), Result-cue paragraph (fal...",True
3,3,512,"[Opening (fallback), Result-cue paragraph (fal...",True
4,4,512,"[Opening (fallback), Result-cue paragraph (fal...",True


Prepared Samples: 100


In [22]:
assert len(holdout_prepared_df) == 100

assert holdout_prepared_df["article_id"].nunique() == 100

assert (
    holdout_prepared_df["article_id"].tolist()
    == holdout_article_ids
)

assert (
    holdout_prepared_df["source_token_count"]
    <= MAX_INPUT_LENGTH
).all()

assert (
    holdout_prepared_df["source_token_count"] > 0
).all()

assert holdout_prepared_df["input_text"].apply(
    lambda text: isinstance(text, str) and bool(text.strip())
).all()

print("Prepared Holdout Samples:", len(holdout_prepared_df))

print(
    "Minimum Input Tokens:",
    holdout_prepared_df["source_token_count"].min()
)

print(
    "Mean Input Tokens:",
    holdout_prepared_df["source_token_count"].mean()
)

print(
    "Maximum Input Tokens:",
    holdout_prepared_df["source_token_count"].max()
)

print("Holdout Preparation Verification: PASS")

Prepared Holdout Samples: 100
Minimum Input Tokens: 511
Mean Input Tokens: 511.88
Maximum Input Tokens: 512
Holdout Preparation Verification: PASS


In [23]:
BATCH_SIZE = 4

holdout_batches = []

for start in range(
    0,
    len(holdout_prepared_df),
    BATCH_SIZE
):

    batch_df = holdout_prepared_df.iloc[
        start:start + BATCH_SIZE
    ]

    encoded_batch = tokenizer(
        batch_df["input_text"].tolist(),
        max_length=MAX_INPUT_LENGTH,
        truncation=False,
        padding=True,
        return_tensors="pt"
    )

    holdout_batches.append({
        "sample_ids": batch_df["sample_id"].tolist(),
        "article_ids": batch_df["article_id"].tolist(),
        "input_ids": encoded_batch["input_ids"],
        "attention_mask": encoded_batch["attention_mask"]
    })

assert len(holdout_batches) == 25

assert sum(
    len(batch["sample_ids"])
    for batch in holdout_batches
) == 100

assert all(
    batch["input_ids"].shape[1] <= MAX_INPUT_LENGTH
    for batch in holdout_batches
)

print("Total Batches:", len(holdout_batches))
print("Batch Size:", BATCH_SIZE)
print("First Batch Sample IDs:", holdout_batches[0]["sample_ids"])
print("Input IDs Shape:", holdout_batches[0]["input_ids"].shape)
print("Attention Mask Shape:", holdout_batches[0]["attention_mask"].shape)

Total Batches: 25
Batch Size: 4
First Batch Sample IDs: [0, 1, 2, 3]
Input IDs Shape: torch.Size([4, 512])
Attention Mask Shape: torch.Size([4, 512])


/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/transformers/tokenization_utils_base.py:2368: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [24]:
from contextlib import nullcontext

def generate_holdout_outputs(eval_model, disable_lora=False):

    eval_model.eval()

    generated_records = []

    adapter_context = (
        eval_model.disable_adapter()
        if disable_lora
        else nullcontext()
    )

    with adapter_context, torch.inference_mode():

        for batch in holdout_batches:

            inputs = {
                key: batch[key].to(device)
                for key in ["input_ids", "attention_mask"]
            }

            with torch.autocast(
                "cuda",
                dtype=torch.bfloat16,
                enabled=USE_BF16
            ):

                generated = eval_model.generate(
                    **inputs,
                    max_length=GENERATION_MAX_LENGTH,
                    max_new_tokens=None,
                    num_beams=GENERATION_NUM_BEAMS,
                    repetition_penalty=REPETITION_PENALTY,
                    no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                    do_sample=DO_SAMPLE
                )

            generated = generated.cpu().tolist()

            for sample_id, article_id, token_ids in zip(
                batch["sample_ids"],
                batch["article_ids"],
                generated
            ):

                generated_records.append({
                    "sample_id": sample_id,
                    "article_id": article_id,
                    "generated_ids": token_ids
                })

    return generated_records

In [26]:
assert hashlib.sha256(
    final_config_path.read_bytes()
).hexdigest() == final_config_hash

set_seed(SEED)

zero_shot_outputs = generate_holdout_outputs(
    model,
    disable_lora=True
)

set_seed(SEED)

lora_outputs = generate_holdout_outputs(
    model,
    disable_lora=False
)

assert len(zero_shot_outputs) == 100
assert len(lora_outputs) == 100

assert [
    row["article_id"]
    for row in zero_shot_outputs
] == holdout_article_ids

assert [
    row["article_id"]
    for row in lora_outputs
] == holdout_article_ids

print("Zero-Shot Outputs:", len(zero_shot_outputs))
print("LoRA Outputs:", len(lora_outputs))

print(
    "Total Generations:",
    len(zero_shot_outputs) + len(lora_outputs)
)

print("Holdout Generation: PASS")

Zero-Shot Outputs: 100
LoRA Outputs: 100
Total Generations: 200
Holdout Generation: PASS


In [27]:
special_token_ids = set(tokenizer.all_special_ids)

eos_token_id = tokenizer.eos_token_id

forced_eos_token_id = model.generation_config.forced_eos_token_id

assert forced_eos_token_id is None

def analyze_holdout_generation(token_ids):

    eos_position = next(
        (
            i
            for i, token_id in enumerate(token_ids)
            if i > 0 and token_id == eos_token_id
        ),
        None
    )

    effective_ids = (
        token_ids[:eos_position + 1]
        if eos_position is not None
        else token_ids
    )

    content_ids = [
        token_id
        for token_id in effective_ids[1:]
        if token_id not in special_token_ids
    ]

    generated_tokens = len(content_ids)

    eos_before_limit = (
        eos_position is not None
        and eos_position < GENERATION_MAX_LENGTH - 1
    )

    reached_max_length = (
        len(effective_ids) >= GENERATION_MAX_LENGTH
    )

    generated_text = tokenizer.decode(
        effective_ids,
        skip_special_tokens=True
    )

    trigrams = list(zip(
        content_ids,
        content_ids[1:],
        content_ids[2:]
    ))

    repeated_trigram_count = (
        len(trigrams) - len(set(trigrams))
    )

    repeated_trigram_ratio = (
        repeated_trigram_count / len(trigrams)
        if trigrams
        else 0.0
    )

    return {
        "generated_tokens": generated_tokens,
        "eos_generated": eos_position is not None,
        "eos_position": eos_position,
        "eos_before_limit": eos_before_limit,
        "reached_max_length": reached_max_length,
        "empty_output": not generated_text.strip(),
        "repeated_trigram_count": repeated_trigram_count,
        "repeated_trigram_ratio": repeated_trigram_ratio,
        "generated_text": generated_text
    }

In [29]:
holdout_prediction_records = []

for model_name, outputs in [
    ("zero_shot", zero_shot_outputs),
    ("lora", lora_outputs)
]:

    for output in outputs:

        generation_analysis = analyze_holdout_generation(
            output["generated_ids"]
        )

        holdout_prediction_records.append({
            "sample_id": output["sample_id"],
            "article_id": output["article_id"],
            "model": model_name,
            "config": "C",
            **generation_analysis
        })

final_holdout_predictions_df = pd.DataFrame(
    holdout_prediction_records
)

assert len(final_holdout_predictions_df) == 200

assert not final_holdout_predictions_df.duplicated(
    subset=["article_id", "model"]
).any()

assert final_holdout_predictions_df.groupby(
    "model"
).size().eq(100).all()

display(final_holdout_predictions_df[[
    "sample_id",
    "model",
    "config",
    "generated_tokens",
    "eos_generated",
    "eos_before_limit",
    "reached_max_length",
    "repeated_trigram_ratio",
    "empty_output"
]])

print("Total Records:", len(final_holdout_predictions_df))

,sample_id,model,config,generated_tokens,eos_generated,eos_before_limit,reached_max_length,repeated_trigram_ratio,empty_output
0,0,zero_shot,C,61,True,True,False,0.0,False
1,1,zero_shot,C,23,True,True,False,0.0,False
2,2,zero_shot,C,44,True,True,False,0.0,False
3,3,zero_shot,C,66,True,True,False,0.0,False
4,4,zero_shot,C,22,True,True,False,0.0,False
...,...,...,...,...,...,...,...,...,...
195,95,lora,C,255,False,False,True,0.0,False
196,96,lora,C,100,True,True,False,0.0,False
197,97,lora,C,255,False,False,True,0.0,False
198,98,lora,C,255,False,False,True,0.0,False


Total Records: 200


In [30]:
holdout_generation_summary = (
    final_holdout_predictions_df
    .groupby("model")
    .agg(
        sample_count=("sample_id", "count"),
        avg_generated_tokens=("generated_tokens", "mean"),
        median_generated_tokens=("generated_tokens", "median"),
        max_generated_tokens=("generated_tokens", "max"),
        early_eos_rate=("eos_before_limit", "mean"),
        max_length_rate=("reached_max_length", "mean"),
        repeated_trigram_ratio=("repeated_trigram_ratio", "mean"),
        empty_output_count=("empty_output", "sum"),
        empty_output_rate=("empty_output", "mean")
    )
)

rate_columns = [
    "early_eos_rate",
    "max_length_rate",
    "empty_output_rate"
]

holdout_generation_summary[rate_columns] *= 100

display(holdout_generation_summary.T)

model,lora,zero_shot
sample_count,100.000000,100.0000
avg_generated_tokens,213.930000,26.1100
median_generated_tokens,233.000000,26.5000
max_generated_tokens,255.000000,66.0000
early_eos_rate,60.000000,100.0000
max_length_rate,40.000000,0.0000
repeated_trigram_ratio,0.000559,0.0004
empty_output_count,0.000000,0.0000
empty_output_rate,0.000000,0.0000


In [31]:
import evaluate

rouge_metric = evaluate.load("rouge")

holdout_references = list(
    holdout_dataset["summary"]
)

assert len(holdout_references) == 100

assert all(
    isinstance(reference, str) and bool(reference.strip())
    for reference in holdout_references
)

print("Reference Count:", len(holdout_references))
print("ROUGE Metric: Ready")

Reference Count: 100
ROUGE Metric: Ready


In [32]:
def compute_holdout_rouge(predictions, references):

    assert len(predictions) == len(references)

    random_state = np.random.get_state()

    try:

        np.random.seed(SEED)

        scores = rouge_metric.compute(
            predictions=predictions,
            references=references,
            use_stemmer=True
        )

    finally:

        np.random.set_state(random_state)

    return {
        "rouge1": scores["rouge1"] * 100,
        "rouge2": scores["rouge2"] * 100,
        "rougeL": scores["rougeL"] * 100
    }

In [34]:
holdout_rouge_records = []

for model_name in ["zero_shot", "lora"]:

    model_predictions = (
        final_holdout_predictions_df[
            final_holdout_predictions_df["model"] == model_name
        ]
        .sort_values("sample_id")
        .reset_index(drop=True)
    )

    assert model_predictions["article_id"].tolist() == holdout_article_ids

    predictions = model_predictions[
        "generated_text"
    ].tolist()

    assert len(predictions) == 100

    rouge_scores = compute_holdout_rouge(
        predictions,
        holdout_references
    )

    holdout_rouge_records.append({
        "model": model_name,
        "config": "C",
        "sample_count": len(predictions),
        **rouge_scores
    })

final_holdout_rouge_df = pd.DataFrame(
    holdout_rouge_records
)

display(final_holdout_rouge_df)

,model,config,sample_count,rouge1,rouge2,rougeL
0,zero_shot,C,100,4.656008,1.084190,3.746519
1,lora,C,100,26.067595,5.503472,14.359853


In [36]:
sample_rouge_records = []

for model_name in ["zero_shot", "lora"]:

    model_predictions = (
        final_holdout_predictions_df[
            final_holdout_predictions_df["model"] == model_name
        ]
        .sort_values("sample_id")
        .reset_index(drop=True)
    )

    assert model_predictions["article_id"].tolist() == holdout_article_ids

    predictions = model_predictions[
        "generated_text"
    ].tolist()

    sample_scores = rouge_metric.compute(
        predictions=predictions,
        references=holdout_references,
        use_stemmer=True,
        use_aggregator=False
    )

    for i, article_id in enumerate(holdout_article_ids):

        sample_rouge_records.append({
            "sample_id": i,
            "article_id": article_id,
            "model": model_name,
            "rouge1": sample_scores["rouge1"][i] * 100,
            "rouge2": sample_scores["rouge2"][i] * 100,
            "rougeL": sample_scores["rougeL"][i] * 100
        })

sample_rouge_df = pd.DataFrame(
    sample_rouge_records
)

assert len(sample_rouge_df) == 200

assert not sample_rouge_df.duplicated(
    subset=["article_id", "model"]
).any()

assert sample_rouge_df[
    ["rouge1", "rouge2", "rougeL"]
].apply(lambda column: column.between(0, 100)).all().all()

display(sample_rouge_df)

print("Total ROUGE Records:", len(sample_rouge_df))

,sample_id,article_id,model,rouge1,rouge2,rougeL
0,0,7e4d0ceecbf944c2fabef6062387d53d51a6fed5399333...,zero_shot,8.000000,1.342282,6.666667
1,1,1dc13fc87e69b8500bf9c050f025fa50d997d41d3f5e68...,zero_shot,5.263158,1.587302,4.210526
2,2,b7fd518a412a7c1eb5d3613e90ede6394290128192e0eb...,zero_shot,8.153477,1.445783,4.316547
3,3,225b7cdde1a04a32b78d1d2711d00aadc067efc22ab497...,zero_shot,11.989101,2.739726,6.539510
4,4,5e6862c26f0e75036f883e2a8552a1c48f142060a29f0e...,zero_shot,8.661417,2.380952,7.874016
...,...,...,...,...,...,...
195,95,d1101e7da247fde6daea0fa8832670937305180e8bd5c2...,lora,32.954545,6.463878,18.181818
196,96,ca9eecf06a6486866bb7f874569445452ace917b85f05e...,lora,15.625000,2.745098,8.593750
197,97,31275d6e810ee64860c3920e39a095e3c00d8823fa946d...,lora,28.985507,6.237006,15.320911
198,98,7c659e4af7dc8b5991b680f03e07d5d7aba5003a3a3e5a...,lora,30.198020,4.477612,16.336634


Total ROUGE Records: 200


In [37]:
final_holdout_predictions_df = (
    final_holdout_predictions_df.merge(
        sample_rouge_df,
        on=["sample_id", "article_id", "model"],
        how="inner",
        validate="one_to_one"
    )
)

assert len(final_holdout_predictions_df) == 200

display(final_holdout_predictions_df[[
    "sample_id",
    "model",
    "generated_tokens",
    "eos_before_limit",
    "reached_max_length",
    "repeated_trigram_ratio",
    "rouge1",
    "rouge2",
    "rougeL"
]])

,sample_id,model,generated_tokens,eos_before_limit,reached_max_length,repeated_trigram_ratio,rouge1,rouge2,rougeL
0,0,zero_shot,61,True,False,0.0,8.000000,1.342282,6.666667
1,1,zero_shot,23,True,False,0.0,5.263158,1.587302,4.210526
2,2,zero_shot,44,True,False,0.0,8.153477,1.445783,4.316547
3,3,zero_shot,66,True,False,0.0,11.989101,2.739726,6.539510
4,4,zero_shot,22,True,False,0.0,8.661417,2.380952,7.874016
...,...,...,...,...,...,...,...,...,...
195,95,lora,255,False,True,0.0,32.954545,6.463878,18.181818
196,96,lora,100,True,False,0.0,15.625000,2.745098,8.593750
197,97,lora,255,False,True,0.0,28.985507,6.237006,15.320911
198,98,lora,255,False,True,0.0,30.198020,4.477612,16.336634


In [38]:
final_holdout_predictions_df.to_csv(
    reports_dir / "final_holdout_predictions.csv",
    index=False
)

print("Prediction Records:", len(final_holdout_predictions_df))
print("Final Holdout Predictions: Saved Successfully")

Prediction Records: 200
Final Holdout Predictions: Saved Successfully


In [39]:
final_holdout_metrics_df = (
    final_holdout_rouge_df.merge(
        holdout_generation_summary.reset_index(),
        on=["model", "sample_count"],
        how="inner",
        validate="one_to_one"
    )
)

assert len(final_holdout_metrics_df) == 2

display(final_holdout_metrics_df.T)

,0,1
model,zero_shot,lora
config,C,C
sample_count,100,100
rouge1,4.656008,26.067595
rouge2,1.08419,5.503472
rougeL,3.746519,14.359853
avg_generated_tokens,26.11,213.93
median_generated_tokens,26.5,233.0
max_generated_tokens,66,255
early_eos_rate,100.0,60.0


In [40]:
metric_mapping = {
    "ROUGE-1": "rouge1",
    "ROUGE-2": "rouge2",
    "ROUGE-L": "rougeL",
    "Avg Generated Tokens": "avg_generated_tokens",
    "Early EOS Rate (%)": "early_eos_rate",
    "Max Length Rate (%)": "max_length_rate",
    "Repeated Trigram Ratio": "repeated_trigram_ratio",
    "Empty Outputs": "empty_output_count",
    "Empty Output Rate (%)": "empty_output_rate"
}

metrics_index = final_holdout_metrics_df.set_index("model")

comparison_records = []

for metric_name, metric_column in metric_mapping.items():

    zero_shot_value = metrics_index.loc[
        "zero_shot", metric_column
    ]

    lora_value = metrics_index.loc[
        "lora", metric_column
    ]

    comparison_records.append({
        "metric": metric_name,
        "zero_shot": zero_shot_value,
        "lora": lora_value,
        "difference": lora_value - zero_shot_value
    })

final_holdout_comparison_df = pd.DataFrame(
    comparison_records
)

display(final_holdout_comparison_df.round(6))

,metric,zero_shot,lora,difference
0,ROUGE-1,4.656008,26.067595,21.411587
1,ROUGE-2,1.084190,5.503472,4.419283
2,ROUGE-L,3.746519,14.359853,10.613334
3,Avg Generated Tokens,26.110000,213.930000,187.820000
4,Early EOS Rate (%),100.000000,60.000000,-40.000000
5,Max Length Rate (%),0.000000,40.000000,40.000000
6,Repeated Trigram Ratio,0.000400,0.000559,0.000159
7,Empty Outputs,0.000000,0.000000,0.000000
8,Empty Output Rate (%),0.000000,0.000000,0.000000


In [41]:
final_holdout_comparison_df.to_csv(
    reports_dir / "final_holdout_metrics.csv",
    index=False
)

print("Total Metrics:", len(final_holdout_comparison_df))

print(
    "Predictions File Exists:",
    (reports_dir / "final_holdout_predictions.csv").exists()
)

print(
    "Metrics File Exists:",
    (reports_dir / "final_holdout_metrics.csv").exists()
)

print("Final Holdout Metrics: Saved Successfully")

Total Metrics: 9
Predictions File Exists: True
Metrics File Exists: True
Final Holdout Metrics: Saved Successfully


In [42]:
final_holdout_comparison_df.to_csv(
    reports_dir / "final_holdout_metrics.csv",
    index=False
)

print("Final Holdout Metrics: Saved Successfully")

Final Holdout Metrics: Saved Successfully


In [43]:
dev_decoding_df = pd.read_csv(
    reports_dir / "decoding_experiment.csv"
)

dev_lora_metrics = dev_decoding_df.loc[
    (dev_decoding_df["model"] == "lora")
    & (dev_decoding_df["config"] == "C")
].iloc[0]

holdout_lora_metrics = final_holdout_metrics_df.loc[
    final_holdout_metrics_df["model"] == "lora"
].iloc[0]

display(dev_lora_metrics)

model                                lora
config                                  C
sample_count                           20
rouge1                          26.044892
rouge2                           5.502064
rougeL                          14.671474
avg_tokens                         201.25
mean_distinct_1                   0.43741
mean_distinct_2                  0.760291
repeated_trigram_ratio                0.0
total_repeated_trigram_count            0
mean_repeated_trigram_count           0.0
max_trigram_frequency                   1
max_length_rate                      25.0
eos_before_limit_rate                75.0
empty_output_rate                     0.0
Name: 2, dtype: object

In [45]:
generalization_records = []

for metric in ["rouge1", "rouge2", "rougeL"]:

    dev_value = dev_lora_metrics[metric]

    holdout_value = holdout_lora_metrics[metric]

    generalization_records.append({
        "metric": metric,
        "dev_20": dev_value,
        "holdout_100": holdout_value,
        "difference": holdout_value - dev_value
    })

generalization_df = pd.DataFrame(
    generalization_records
)

display(generalization_df.round(6))

,metric,dev_20,holdout_100,difference
0,rouge1,26.044892,26.067595,0.022703
1,rouge2,5.502064,5.503472,0.001409
2,rougeL,14.671474,14.359853,-0.311621


In [46]:
paired_rouge_df = (
    sample_rouge_df.pivot(
        index=["sample_id", "article_id"],
        columns="model",
        values="rougeL"
    )
    .reset_index()
    .sort_values("sample_id")
    .reset_index(drop=True)
)

paired_rouge_df["rougeL_difference"] = (
    paired_rouge_df["lora"]
    - paired_rouge_df["zero_shot"]
)

assert len(paired_rouge_df) == 100

assert paired_rouge_df["article_id"].tolist() == holdout_article_ids

assert not paired_rouge_df[
    ["zero_shot", "lora"]
].isna().any().any()

display(paired_rouge_df.head(10))

print("Paired Samples:", len(paired_rouge_df))

model,sample_id,article_id,lora,zero_shot,rougeL_difference
0,0,7e4d0ceecbf944c2fabef6062387d53d51a6fed5399333...,18.581907,6.666667,11.915240
1,1,1dc13fc87e69b8500bf9c050f025fa50d997d41d3f5e68...,15.605749,4.210526,11.395223
2,2,b7fd518a412a7c1eb5d3613e90ede6394290128192e0eb...,14.801444,4.316547,10.484897
3,3,225b7cdde1a04a32b78d1d2711d00aadc067efc22ab497...,13.617021,6.539510,7.077512
4,4,5e6862c26f0e75036f883e2a8552a1c48f142060a29f0e...,17.365269,7.874016,9.491254
5,5,674821a07585f2293717a2e9ebb27bfae1073a9cabf79f...,14.802065,5.813953,8.988112
6,6,24fd93e224dbeb9802f491390c1ed7a523b74f49f828bb...,16.962525,3.000000,13.962525
7,7,43a536b9eca8c68bb9be7b150aca6c1ea3148293ab86b4...,11.707317,0.641026,11.066291
8,8,746810a3e0250908372e4c1b801806382e85fbab4b8695...,14.689266,0.000000,14.689266
9,9,3702f6077d9eed894a04595fc7c9317066bea5c8bbdf21...,12.406015,4.597701,7.808314


Paired Samples: 100


In [47]:
paired_differences = paired_rouge_df[
    "rougeL_difference"
].to_numpy()

observed_difference = paired_differences.mean()

print("Sample Count:", len(paired_differences))
print("Observed ROUGE-L Difference:", observed_difference)

print(
    "LoRA Wins:",
    (paired_differences > 0).sum()
)

print(
    "Zero-Shot Wins:",
    (paired_differences < 0).sum()
)

print(
    "Ties:",
    (paired_differences == 0).sum()
)

Sample Count: 100
Observed ROUGE-L Difference: 10.605736731671723
LoRA Wins: 98
Zero-Shot Wins: 2
Ties: 0


In [48]:
rng = np.random.default_rng(42)

example_indices = rng.choice(
    5,
    size=5,
    replace=True
)

print(example_indices)

[0 3 3 2 2]


In [49]:
BOOTSTRAP_ITERATIONS = 1000

rng = np.random.default_rng(SEED)

sample_count = len(paired_differences)

bootstrap_differences = []

for _ in range(BOOTSTRAP_ITERATIONS):

    sampled_indices = rng.choice(
        sample_count,
        size=sample_count,
        replace=True
    )

    sampled_differences = paired_differences[
        sampled_indices
    ]

    bootstrap_difference = sampled_differences.mean()

    bootstrap_differences.append(
        bootstrap_difference
    )

bootstrap_differences = np.array(
    bootstrap_differences
)

assert len(bootstrap_differences) == 1000

print("Bootstrap Iterations:", len(bootstrap_differences))

print(
    "Bootstrap Mean Difference:",
    bootstrap_differences.mean()
)

Bootstrap Iterations: 1000
Bootstrap Mean Difference: 10.59620521743611


In [51]:
confidence_level = 0.95

ci_lower, ci_upper = np.percentile(
    bootstrap_differences,
    [2.5, 97.5]
)

bootstrap_mean = bootstrap_differences.mean()

bootstrap_standard_error = bootstrap_differences.std(
    ddof=1
)

print("Observed Difference:", observed_difference)
print("Bootstrap Mean Difference:", bootstrap_mean)
print("Bootstrap Standard Error:", bootstrap_standard_error)
print("95% CI Lower:", ci_lower)
print("95% CI Upper:", ci_upper)
print("Confidence Interval Contains Zero:", ci_lower <= 0 <= ci_upper)

Observed Difference: 10.605736731671723
Bootstrap Mean Difference: 10.59620521743611
Bootstrap Standard Error: 0.3453147798869676
95% CI Lower: 9.940739218357008
95% CI Upper: 11.230661392152019
Confidence Interval Contains Zero: False


In [52]:
confidence_level = 0.95

ci_lower, ci_upper = np.percentile(
    bootstrap_differences,
    [2.5, 97.5]
)

bootstrap_mean = bootstrap_differences.mean()

bootstrap_standard_error = bootstrap_differences.std(
    ddof=1
)

print("Observed Difference:", observed_difference)
print("Bootstrap Mean Difference:", bootstrap_mean)
print("Bootstrap Standard Error:", bootstrap_standard_error)
print("95% CI Lower:", ci_lower)
print("95% CI Upper:", ci_upper)
print("Confidence Interval Contains Zero:", ci_lower <= 0 <= ci_upper)

Observed Difference: 10.605736731671723
Bootstrap Mean Difference: 10.59620521743611
Bootstrap Standard Error: 0.3453147798869676
95% CI Lower: 9.940739218357008
95% CI Upper: 11.230661392152019
Confidence Interval Contains Zero: False


In [53]:
holdout_bootstrap_df = pd.DataFrame([{
    "metric": "rougeL",
    "comparison": "lora_minus_zero_shot",
    "config": "C",
    "sample_count": len(paired_differences),
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "seed": SEED,
    "bootstrap_method": "paired_percentile_bootstrap",
    "confidence_level": confidence_level,
    "observed_difference": observed_difference,
    "aggregated_rougeL_difference": (
        metrics_index.loc["lora", "rougeL"]
        - metrics_index.loc["zero_shot", "rougeL"]
    ),
    "bootstrap_mean_difference": bootstrap_mean,
    "bootstrap_standard_error": bootstrap_standard_error,
    "ci_lower": ci_lower,
    "ci_upper": ci_upper,
    "ci_contains_zero": bool(ci_lower <= 0 <= ci_upper),
    "lora_wins": int((paired_differences > 0).sum()),
    "zero_shot_wins": int((paired_differences < 0).sum()),
    "ties": int((paired_differences == 0).sum())
}])

assert ci_lower <= ci_upper
assert len(paired_differences) == 100
assert np.isfinite(bootstrap_differences).all()

holdout_bootstrap_df.to_csv(
    reports_dir / "holdout_bootstrap.csv",
    index=False
)

display(holdout_bootstrap_df.T)

print("Bootstrap Report: Saved Successfully")

,0
metric,rougeL
comparison,lora_minus_zero_shot
config,C
sample_count,100
bootstrap_iterations,1000
seed,42
bootstrap_method,paired_percentile_bootstrap
confidence_level,0.95
observed_difference,10.605737
aggregated_rougeL_difference,10.613334


Bootstrap Report: Saved Successfully


In [54]:
FACTUALITY_SAMPLE_SIZE = 10

sorted_holdout_df = (
    holdout_prepared_df[["sample_id", "article_id"]]
    .sort_values("article_id")
    .reset_index(drop=True)
)

rng = np.random.default_rng(SEED)

selected_indices = rng.choice(
    len(sorted_holdout_df),
    size=FACTUALITY_SAMPLE_SIZE,
    replace=False
)

factuality_samples_df = (
    sorted_holdout_df.iloc[selected_indices]
    .sort_values("sample_id")
    .reset_index(drop=True)
)

factuality_sample_ids = (
    factuality_samples_df["sample_id"].tolist()
)

assert len(factuality_samples_df) == 10
assert factuality_samples_df["article_id"].nunique() == 10

assert set(
    factuality_samples_df["article_id"]
).issubset(holdout_ids)

display(factuality_samples_df)

print("Selected Sample IDs:", factuality_sample_ids)
print("Selected Articles:", len(factuality_samples_df))
print("Factuality Sample Selection: PASS")

,sample_id,article_id
0,18,d07baa11f86a0b36b6422847bd20bc8b5c20b02150fa75...
1,31,f12fc5a5588f63cffddd368425034c40736f8dcf5b3125...
2,42,2d35fc30152d01481c726b2d3f9c26523bb0b8d90112fb...
3,50,a1429cc8dedc90d48fa9a2294e9586a168d7cd01f0640d...
4,55,1339cf0806aac58c617d2e41985ba220acdbd4a680cf8f...
5,62,71c1fef70e84f8b2a4bc05c9de4e1620a1503847839c4e...
6,79,f5d5ba94222d329268fa3f32ab948e48fdb48ce104a2b9...
7,82,17dc3ce586826abba29de3893ae3ac9c8346f7a52bcc73...
8,93,b6cbc431bf1a211252ffafa2c9138423d1ed8544e27751...
9,94,b91ae9fe362736529a9479c5a2ce277226e3f2df1f535a...


Selected Sample IDs: [18, 31, 42, 50, 55, 62, 79, 82, 93, 94]
Selected Articles: 10
Factuality Sample Selection: PASS


In [55]:
factuality_sample_manifest_path = (
    reports_dir / "factuality_sample_manifest.csv"
)

factuality_samples_df["seed"] = SEED
factuality_samples_df["config"] = "C"
factuality_samples_df["split"] = "final_holdout"

if factuality_sample_manifest_path.exists():

    existing_samples = pd.read_csv(
        factuality_sample_manifest_path
    )

    assert existing_samples.equals(factuality_samples_df)

else:

    factuality_samples_df.to_csv(
        factuality_sample_manifest_path,
        index=False
    )

print("Selected Articles:", len(factuality_samples_df))
print("Models Per Article:", 2)
print("Total Summaries:", len(factuality_samples_df) * 2)
print("Sample Manifest: Saved Successfully")

Selected Articles: 10
Models Per Article: 2
Total Summaries: 20
Sample Manifest: Saved Successfully


In [56]:
from tempfile import mkdtemp

review_dir = Path(
    mkdtemp(prefix="healthbrief_factuality_")
)

for sample_id in factuality_sample_ids:

    sample = holdout_dataset[sample_id]

    assert create_article_id(
        sample["article"]
    ) == holdout_article_ids[sample_id]

    review_text = (
        f"SAMPLE ID: {sample_id}\n\n"
        f"TITLE:\n{sample['title']}\n\n"
        f"REFERENCE SUMMARY:\n{sample['summary']}\n\n"
        f"FULL ARTICLE:\n{sample['article']}"
    )

    review_path = review_dir / f"sample_{sample_id}.txt"

    review_path.write_text(
        review_text,
        encoding="utf-8"
    )

print("Review Directory:", review_dir)
print("Prepared Articles:", len(list(review_dir.glob("*.txt"))))

Review Directory: /tmp/healthbrief_factuality_120z0195
Prepared Articles: 10


In [57]:
key_finding_records = []

for sample_id in factuality_sample_ids:

    for finding_id in range(1, 4):

        key_finding_records.append({
            "sample_id": sample_id,
            "article_id": holdout_article_ids[sample_id],
            "finding_id": finding_id,
            "key_finding": "",
            "evidence": ""
        })

key_findings_df = pd.DataFrame(
    key_finding_records
)

assert len(key_findings_df) == 30

assert key_findings_df.groupby(
    "sample_id"
).size().eq(3).all()

display(key_findings_df.head(9))

print("Total Key Findings:", len(key_findings_df))

,sample_id,article_id,finding_id,key_finding,evidence
0,18,d07baa11f86a0b36b6422847bd20bc8b5c20b02150fa75...,1,,
1,18,d07baa11f86a0b36b6422847bd20bc8b5c20b02150fa75...,2,,
2,18,d07baa11f86a0b36b6422847bd20bc8b5c20b02150fa75...,3,,
3,31,f12fc5a5588f63cffddd368425034c40736f8dcf5b3125...,1,,
4,31,f12fc5a5588f63cffddd368425034c40736f8dcf5b3125...,2,,
5,31,f12fc5a5588f63cffddd368425034c40736f8dcf5b3125...,3,,
6,42,2d35fc30152d01481c726b2d3f9c26523bb0b8d90112fb...,1,,
7,42,2d35fc30152d01481c726b2d3f9c26523bb0b8d90112fb...,2,,
8,42,2d35fc30152d01481c726b2d3f9c26523bb0b8d90112fb...,3,,


Total Key Findings: 30


In [58]:
sample_id = factuality_sample_ids[0]

sample = holdout_dataset[sample_id]

print("SAMPLE ID:", sample_id)

print("\nTITLE:")
print(sample["title"])

print("\nREFERENCE SUMMARY:")
print(sample["summary"])

print("\nFULL ARTICLE PATH:")
print(review_dir / f"sample_{sample_id}.txt")

SAMPLE ID: 18

TITLE:
Huntingtin’s spherical solenoid structure enables polyglutamine tract-dependent modulation of its structure and function

REFERENCE SUMMARY:
Huntington’s disease is an inherited disorder that occurs in adulthood and sometimes in children . It causes progressive damage to the brain and people with the condition develop memory loss , movement difficulties , confusion , and other symptoms of mental decline . Eventually , the disease leads to death . Mutations in the gene that encodes a protein called huntingtin cause Huntington’s disease . Individuals who inherit just one copy of the mutated gene develop the condition . No treatments currently exist that can slow or stop disease progression . Genetic and molecular studies are beginning to shed light on how mutations in the gene encoding huntingtin cause the disease . Normally , the protein has a section near its tail end made up of the amino acid glutamine repeated around 23 times . Mutations that increase the number

In [60]:
combined_review_path = reports_dir / "all_factuality_articles.txt"

combined_texts = []

for sample_id in factuality_sample_ids:

    file_path = review_dir / f"sample_{sample_id}.txt"

    combined_texts.append(
        file_path.read_text(encoding="utf-8")
    )

combined_review_path.write_text(
    ("\n\n" + "=" * 100 + "\n\n").join(combined_texts),
    encoding="utf-8"
)

assert len(combined_texts) == 10
assert combined_review_path.exists()

print("Total Articles:", len(combined_texts))
print("File Path:", combined_review_path)
print("Report Saved Successfully")

Total Articles: 10
File Path: /home/pasali/Desktop/Data/AI-Engineer/Hafta-7/HealthBrief/reports/all_factuality_articles.txt
Report Saved Successfully


In [62]:
key_findings_df = pd.read_csv(
    reports_dir / "final_key_findings.csv"
)

assert len(key_findings_df) == 30

assert key_findings_df["article_id"].nunique() == 10

assert key_findings_df.groupby(
    "sample_id"
).size().eq(3).all()

assert not key_findings_df.duplicated(
    subset=["article_id", "finding_id"]
).any()

assert set(
    zip(
        key_findings_df["sample_id"],
        key_findings_df["article_id"]
    )
) == set(
    zip(
        factuality_samples_df["sample_id"],
        factuality_samples_df["article_id"]
    )
)

assert key_findings_df[
    ["key_finding", "evidence"]
].notna().all().all()

assert key_findings_df[
    ["key_finding", "evidence"]
].apply(
    lambda column: column.str.strip().ne("")
).all().all()

display(key_findings_df)

print("Total Articles:", key_findings_df["article_id"].nunique())
print("Total Key Findings:", len(key_findings_df))
print("Key Finding Verification: PASS")

FileNotFoundError: [Errno 2] No such file or directory: '/home/pasali/Desktop/Data/AI-Engineer/Hafta-7/HealthBrief/reports/final_key_findings.csv'

In [63]:
key_findings_df = pd.read_csv(
    reports_dir / "final_key_findings.csv"
)

assert len(key_findings_df) == 30

assert key_findings_df["article_id"].nunique() == 10

assert key_findings_df.groupby(
    "sample_id"
).size().eq(3).all()

assert not key_findings_df.duplicated(
    subset=["article_id", "finding_id"]
).any()

assert set(
    zip(
        key_findings_df["sample_id"],
        key_findings_df["article_id"]
    )
) == set(
    zip(
        factuality_samples_df["sample_id"],
        factuality_samples_df["article_id"]
    )
)

assert key_findings_df[
    ["key_finding", "evidence"]
].notna().all().all()

assert key_findings_df[
    ["key_finding", "evidence"]
].apply(
    lambda column: column.str.strip().ne("")
).all().all()

display(key_findings_df)

print("Total Articles:", key_findings_df["article_id"].nunique())
print("Total Key Findings:", len(key_findings_df))
print("Key Finding Verification: PASS")

,sample_id,article_id,finding_id,key_finding,evidence
0,18,d07baa11f86a0b36b6422847bd20bc8b5c20b02150fa75...,1,Full-length huntingtin adopts a spherical alph...,adopts a spherical α-helical solenoid where th...
1,18,d07baa11f86a0b36b6422847bd20bc8b5c20b02150fa75...,2,The N- and C-terminal regions fold into close ...,the ends of each arm are in close proximity an...
2,18,d07baa11f86a0b36b6422847bd20bc8b5c20b02150fa75...,3,Polyglutamine expansion increases huntingtin a...,the polyglutamine expansion increases α-helica...
3,31,f12fc5a5588f63cffddd368425034c40736f8dcf5b3125...,1,"Unlike DFT1, DFT2 can transmit between Tasmani...","in contrast to DFT1 , DFT2 cells express major..."
4,31,f12fc5a5588f63cffddd368425034c40736f8dcf5b3125...,2,The dominant MHC class I alleles in DFT2 are s...,the most highly expressed MHC class I alleles ...
5,31,f12fc5a5588f63cffddd368425034c40736f8dcf5b3125...,3,MHC class I expression varies among DFT2 tumou...,"MHC class I loss is emerging in vivo , thus DF..."
6,42,2d35fc30152d01481c726b2d3f9c26523bb0b8d90112fb...,1,Mice lacking leptin signaling early in life re...,LepR deficiency in early life led to irreversi...
7,42,2d35fc30152d01481c726b2d3f9c26523bb0b8d90112fb...,2,Restoring leptin receptors in adulthood rescue...,the poor development of arcuate nucleus neural...
8,42,2d35fc30152d01481c726b2d3f9c26523bb0b8d90112fb...,3,Adult leptin receptor restoration did not full...,"persistent reduction in hypothalamic Pomc , Ca..."
9,50,a1429cc8dedc90d48fa9a2294e9586a168d7cd01f0640d...,1,"In Arabidopsis, P4R2 RNA precursors are mostly...",P4R2 RNAs are mostly 26-45 nt […] that accumul...


Total Articles: 10
Total Key Findings: 30
Key Finding Verification: PASS


In [64]:
factuality_predictions_df = (
    final_holdout_predictions_df.loc[
        final_holdout_predictions_df["sample_id"].isin(
            factuality_sample_ids
        )
        & final_holdout_predictions_df["config"].eq("C")
    ]
    .sort_values(["sample_id", "model"])
    .reset_index(drop=True)
)

assert len(factuality_predictions_df) == 20

assert factuality_predictions_df.groupby(
    "model"
).size().eq(10).all()

assert not factuality_predictions_df.duplicated(
    subset=["article_id", "model"]
).any()

factuality_review_texts = []

for sample_id in factuality_sample_ids:

    source = holdout_prepared_df.loc[
        holdout_prepared_df["sample_id"] == sample_id
    ].iloc[0]

    input_ids = tokenizer(
        source["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False
    )["input_ids"]

    assert len(input_ids) == source["source_token_count"]
    assert len(input_ids) <= MAX_INPUT_LENGTH

    source_text = tokenizer.decode(
        input_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    article_predictions = factuality_predictions_df.loc[
        factuality_predictions_df["sample_id"] == sample_id
    ]

    assert set(article_predictions["model"]) == {
        "zero_shot", "lora"
    }

    assert article_predictions["article_id"].eq(
        source["article_id"]
    ).all()

    review_text = (
        f"SAMPLE ID: {sample_id}\n"
        f"ARTICLE ID: {source['article_id']}\n"
        f"TITLE: {holdout_dataset[sample_id]['title']}\n\n"
        f"INPUT TOKEN COUNT: {len(input_ids)}\n\n"
        f"MODEL INPUT:\n{source_text}\n\n"
    )

    for _, prediction in article_predictions.iterrows():

        review_text += (
            f"MODEL: {prediction['model']}\n"
            f"CONFIG: {prediction['config']}\n"
            f"GENERATED TOKENS: {prediction['generated_tokens']}\n"
            f"EOS GENERATED: {prediction['eos_generated']}\n"
            f"MAX LENGTH REACHED: {prediction['reached_max_length']}\n"
            f"REPEATED TRIGRAM RATIO: "
            f"{prediction['repeated_trigram_ratio']}\n\n"
            f"GENERATED SUMMARY:\n"
            f"{prediction['generated_text']}\n\n"
        )

    factuality_review_texts.append(review_text)

factuality_review_path = (
    reports_dir / "final_factuality_review.txt"
)

factuality_review_path.write_text(
    ("\n" + "=" * 100 + "\n").join(
        factuality_review_texts
    ),
    encoding="utf-8"
)

assert len(factuality_review_texts) == 10
assert factuality_review_path.exists()

print("Selected Articles:", len(factuality_review_texts))
print("Selected Predictions:", len(factuality_predictions_df))
print("Review File:", factuality_review_path)
print("Factuality Review Preparation: PASS")

Selected Articles: 10
Selected Predictions: 20
Review File: /home/pasali/Desktop/Data/AI-Engineer/Hafta-7/HealthBrief/reports/final_factuality_review.txt
Factuality Review Preparation: PASS


In [66]:
final_factuality_df = pd.read_csv(
    reports_dir / "final_factual_consistency.csv"
)

final_coverage_df = pd.read_csv(
    reports_dir / "final_key_finding_coverage.csv"
)

final_scientific_errors_df = pd.read_csv(
    reports_dir / "final_scientific_error_analysis.csv"
)

valid_factuality_labels = {
    "Supported",
    "Contradicted",
    "Not Verifiable",
    "Uninterpretable"
}

assert len(final_factuality_df) == 74
assert len(final_coverage_df) == 60
assert len(final_scientific_errors_df) == 20

assert final_factuality_df[
    "input_grounded"
].isin(valid_factuality_labels).all()

assert final_factuality_df[
    "full_article"
].isin(valid_factuality_labels).all()

assert final_factuality_df.groupby(
    "model"
)["article_id"].nunique().eq(10).all()

assert final_coverage_df.groupby(
    "model"
).size().eq(30).all()

assert set(
    final_factuality_df["article_id"]
) == set(factuality_samples_df["article_id"])

assert not final_factuality_df.duplicated(
    subset=["article_id", "model", "claim_id"]
).any()

display(final_factuality_df[[
    "sample_id",
    "model",
    "claim",
    "input_grounded",
    "full_article"
]])

print("Factuality Records:", len(final_factuality_df))
print("Coverage Records:", len(final_coverage_df))
print("Scientific Error Records:", len(final_scientific_errors_df))
print("Factuality Verification: PASS")

,sample_id,model,claim,input_grounded,full_article
0,18,zero_shot,Polyglutamine expansion increases huntingtin a...,Supported,Supported
1,18,zero_shot,Polyglutamine expansion affects intramolecular...,Supported,Supported
2,18,lora,Polyglutamine expansion in huntingtin causes H...,Supported,Supported
3,18,lora,Polyglutamine expansion increases huntingtin a...,Supported,Supported
4,18,lora,Polyglutamine expands the spherical helical so...,Not Verifiable,Not Verifiable
...,...,...,...,...,...
69,94,lora,Tristetraprolin is regulated by the MyoD mRNA ...,Contradicted,Contradicted
70,94,lora,Tristetraprolin promotes myogenic commitment.,Contradicted,Contradicted
71,94,lora,MyoD phosphorylates mRNA and itself during act...,Contradicted,Contradicted
72,94,lora,An unspecified molecule activates MyoD accumul...,Uninterpretable,Uninterpretable


Factuality Records: 74
Coverage Records: 60
Scientific Error Records: 20
Factuality Verification: PASS


In [67]:
def calculate_factuality_metrics(df, label_column):

    summary = pd.crosstab(
        df["model"],
        df[label_column]
    ).reindex(
        columns=[
            "Supported",
            "Contradicted",
            "Not Verifiable",
            "Uninterpretable"
        ],
        fill_value=0
    )

    summary.columns = [
        "supported",
        "contradicted",
        "not_verifiable",
        "uninterpretable"
    ]

    summary["assessable_claims"] = (
        summary["supported"]
        + summary["contradicted"]
        + summary["not_verifiable"]
    )

    denominator = summary["assessable_claims"].replace(0, np.nan)

    summary["supported_rate"] = (
        summary["supported"] / denominator * 100
    )

    summary["contradicted_rate"] = (
        summary["contradicted"] / denominator * 100
    )

    summary["not_verifiable_rate"] = (
        summary["not_verifiable"] / denominator * 100
    )

    return summary


input_factuality_summary = calculate_factuality_metrics(
    final_factuality_df,
    "input_grounded"
)

full_article_factuality_summary = calculate_factuality_metrics(
    final_factuality_df,
    "full_article"
)

print("INPUT-GROUNDED FACTUALITY")
display(input_factuality_summary.round(4))

print("FULL-ARTICLE FACTUALITY")
display(full_article_factuality_summary.round(4))

INPUT-GROUNDED FACTUALITY


,supported,contradicted,not_verifiable,uninterpretable,assessable_claims,supported_rate,contradicted_rate,not_verifiable_rate
model,,,,,,,,
lora,17,17,14,12,48,35.4167,35.4167,29.1667
zero_shot,8,5,0,1,13,61.5385,38.4615,0.0000


FULL-ARTICLE FACTUALITY


,supported,contradicted,not_verifiable,uninterpretable,assessable_claims,supported_rate,contradicted_rate,not_verifiable_rate
model,,,,,,,,
lora,17,18,13,12,48,35.4167,37.5000,27.0833
zero_shot,8,5,0,1,13,61.5385,38.4615,0.0000


In [68]:
factuality_transitions = (
    final_factuality_df
    .groupby([
        "model",
        "input_grounded",
        "full_article"
    ])
    .size()
    .reset_index(name="claim_count")
)

display(factuality_transitions)

,model,input_grounded,full_article,claim_count
0,lora,Contradicted,Contradicted,17
1,lora,Not Verifiable,Contradicted,1
2,lora,Not Verifiable,Not Verifiable,13
3,lora,Supported,Supported,17
4,lora,Uninterpretable,Uninterpretable,12
5,zero_shot,Contradicted,Contradicted,5
6,zero_shot,Supported,Supported,8
7,zero_shot,Uninterpretable,Uninterpretable,1


In [69]:
coverage_summary = (
    final_coverage_df
    .groupby("model")
    .agg(
        total_findings=("finding_id", "count"),
        covered_findings=("covered", "sum"),
        partial_findings=(
            "coverage_status",
            lambda x: x.eq("partial").sum()
        )
    )
)

coverage_summary["key_finding_coverage_rate"] = (
    coverage_summary["covered_findings"]
    / coverage_summary["total_findings"] * 100
)

scientific_error_summary = (
    final_scientific_errors_df
    .groupby("model")
    .agg(
        summary_count=("sample_id", "count"),
        terminology_errors=("terminology_error", "sum"),
        severe_repetition=("severe_repetition", "sum"),
        incomplete_summaries=("incomplete_summary", "sum"),
        uninterpretable_summaries=("uninterpretable_present", "sum"),
        full_article_contradictions=(
            "full_article_contradiction_present", "sum"
        )
    )
)

display(coverage_summary.T)
display(scientific_error_summary.T)

model,lora,zero_shot
total_findings,30.0,30.0
covered_findings,0.0,0.0
partial_findings,7.0,2.0
key_finding_coverage_rate,0.0,0.0


model,lora,zero_shot
summary_count,10,10
terminology_errors,10,2
severe_repetition,0,0
incomplete_summaries,10,10
uninterpretable_summaries,10,1
full_article_contradictions,8,3


In [70]:
final_factuality_comparison = pd.DataFrame({
    "input_supported_claim_rate":
        input_factuality_summary["supported_rate"],
    "full_article_supported_claim_rate":
        full_article_factuality_summary["supported_rate"],
    "input_contradicted_claims":
        input_factuality_summary["contradicted"],
    "full_article_contradicted_claims":
        full_article_factuality_summary["contradicted"],
    "input_not_verifiable_claims":
        input_factuality_summary["not_verifiable"],
    "full_article_not_verifiable_claims":
        full_article_factuality_summary["not_verifiable"],
    "key_findings_covered":
        coverage_summary["covered_findings"],
    "partial_findings":
        coverage_summary["partial_findings"],
    "key_finding_coverage_rate":
        coverage_summary["key_finding_coverage_rate"],
    "terminology_errors":
        scientific_error_summary["terminology_errors"],
    "severe_repetition":
        scientific_error_summary["severe_repetition"],
    "incomplete_summaries":
        scientific_error_summary["incomplete_summaries"],
    "uninterpretable_summaries":
        scientific_error_summary["uninterpretable_summaries"],
    "summaries_with_contradiction":
        scientific_error_summary["full_article_contradictions"]
})

assert len(final_factuality_comparison) == 2

display(
    final_factuality_comparison
    .reindex(["zero_shot", "lora"])
    .T.round(4)
)

final_factuality_comparison.to_csv(
    reports_dir / "final_factuality_summary.csv",
    index=True
)

print("Final Factuality Summary: Saved Successfully")

model,zero_shot,lora
input_supported_claim_rate,61.5385,35.4167
full_article_supported_claim_rate,61.5385,35.4167
input_contradicted_claims,5.0000,17.0000
full_article_contradicted_claims,5.0000,18.0000
input_not_verifiable_claims,0.0000,14.0000
full_article_not_verifiable_claims,0.0000,13.0000
key_findings_covered,0.0000,0.0000
partial_findings,2.0000,7.0000
key_finding_coverage_rate,0.0000,0.0000
terminology_errors,2.0000,10.0000


Final Factuality Summary: Saved Successfully


In [71]:
reliability_holdout_metrics = (
    final_holdout_metrics_df.set_index("model")
)

reliability_factuality_metrics = (
    final_factuality_comparison
)

reliability_metric_mapping = [
    ("Summarization Quality", "ROUGE-L",
     "rougeL", "holdout", 100),

    ("Repetition", "Repeated Trigram Ratio",
     "repeated_trigram_ratio", "holdout", 100),

    ("Termination", "Early EOS Rate (%)",
     "early_eos_rate", "holdout", 100),

    ("Factuality", "Input Supported Claim Rate (%)",
     "input_supported_claim_rate", "factuality", 10),

    ("Factuality", "Full-Article Supported Claim Rate (%)",
     "full_article_supported_claim_rate", "factuality", 10),

    ("Contradictions", "Explicit Contradiction Count",
     "full_article_contradicted_claims", "factuality", 10),

    ("Information Coverage", "Key Finding Coverage (%)",
     "key_finding_coverage_rate", "factuality", 10),

    ("Output Quality", "Empty Output Rate (%)",
     "empty_output_rate", "holdout", 100),

    ("Output Quality", "Uninterpretable Summary Count",
     "uninterpretable_summaries", "factuality", 10)
]

reliability_records = []

for category, metric, column, source, sample_count in reliability_metric_mapping:

    source_df = (
        reliability_holdout_metrics
        if source == "holdout"
        else reliability_factuality_metrics
    )

    reliability_records.append({
        "category": category,
        "metric": metric,
        "zero_shot": source_df.loc["zero_shot", column],
        "lora": source_df.loc["lora", column],
        "sample_count": sample_count,
        "evaluation_source": source
    })

reliability_records.extend([
    {
        "category": "Output Quality",
        "metric": "Invalid Output Rate (%)",
        "zero_shot": np.nan,
        "lora": np.nan,
        "sample_count": 100,
        "evaluation_source": "not_measured_beyond_empty_outputs"
    },
    {
        "category": "Efficiency",
        "metric": "Mean Inference Time (seconds)",
        "zero_shot": np.nan,
        "lora": np.nan,
        "sample_count": 100,
        "evaluation_source": "not_measured_per_model"
    }
])

reliability_scorecard_df = pd.DataFrame(
    reliability_records
)

display(reliability_scorecard_df.round(6))

,category,metric,zero_shot,lora,sample_count,evaluation_source
0,Summarization Quality,ROUGE-L,3.746519,14.359853,100,holdout
1,Repetition,Repeated Trigram Ratio,0.000400,0.000559,100,holdout
2,Termination,Early EOS Rate (%),100.000000,60.000000,100,holdout
3,Factuality,Input Supported Claim Rate (%),61.538462,35.416667,10,factuality
4,Factuality,Full-Article Supported Claim Rate (%),61.538462,35.416667,10,factuality
5,Contradictions,Explicit Contradiction Count,5.000000,18.000000,10,factuality
6,Information Coverage,Key Finding Coverage (%),0.000000,0.000000,10,factuality
7,Output Quality,Empty Output Rate (%),0.000000,0.000000,100,holdout
8,Output Quality,Uninterpretable Summary Count,1.000000,10.000000,10,factuality
9,Output Quality,Invalid Output Rate (%),NaN,NaN,100,not_measured_beyond_empty_outputs


In [72]:
source_df = (
    reliability_holdout_metrics
    if source == "holdout"
    else reliability_factuality_metrics
)

In [73]:
release_decision_df = pd.DataFrame([
    {
        "release_type": "research_prototype",
        "decision": "APPROVED",
        "checkpoint": CHECKPOINT,
        "generation_config": "C",
        "reason": (
            "Completed controlled fine-tuning, independent "
            "holdout evaluation, bootstrap analysis and "
            "documented factuality limitations."
        ),
        "usage_restriction": (
            "Educational and research use only. "
            "Public distribution requires license verification."
        ),
        "production_ready": False
    },
    {
        "release_type": "production_deployment",
        "decision": "REJECTED",
        "checkpoint": CHECKPOINT,
        "generation_config": "C",
        "reason": (
            "Scientific contradictions, low supported claim rate, "
            "terminology errors, incomplete summaries and "
            "insufficient key finding coverage."
        ),
        "usage_restriction": (
            "Not suitable for automated scientific or "
            "clinical decision-making."
        ),
        "production_ready": False
    }
])

reliability_scorecard_df.to_csv(
    reports_dir / "reliability_scorecard.csv",
    index=False
)

release_decision_df.to_csv(
    reports_dir / "release_decision.csv",
    index=False
)

display(release_decision_df.T)

print("Reliability Scorecard: Saved Successfully")
print("Release Decision: Saved Successfully")

,0,1
release_type,research_prototype,production_deployment
decision,APPROVED,REJECTED
checkpoint,checkpoint-96,checkpoint-96
generation_config,C,C
reason,"Completed controlled fine-tuning, independent ...","Scientific contradictions, low supported claim..."
usage_restriction,Educational and research use only. Public dist...,Not suitable for automated scientific or clini...
production_ready,False,False


Reliability Scorecard: Saved Successfully
Release Decision: Saved Successfully


In [74]:
error_cases = [
    {
        "sample_id": 18,
        "model": "lora",
        "claim_id": 5,
        "error_type": "Unsupported Treatment Claim",
        "potential_cause": (
            "The model may associate disease-related structural "
            "findings with therapeutic applications."
        ),
        "user_impact": (
            "Readers may incorrectly assume that a treatment "
            "has been scientifically validated."
        ),
        "future_mitigation": (
            "Evaluate source-grounded training and claim-level "
            "verification of therapeutic assertions."
        )
    },
    {
        "sample_id": 42,
        "model": "zero_shot",
        "claim_id": 1,
        "error_type": "Causality Error",
        "potential_cause": (
            "The model may fail to distinguish early-life receptor "
            "deficiency from adult receptor restoration."
        ),
        "user_impact": (
            "Readers may misunderstand the biological cause "
            "of persistent obesity."
        ),
        "future_mitigation": (
            "Evaluate causal relation preservation and "
            "temporal consistency during summarization."
        )
    },
    {
        "sample_id": 50,
        "model": "lora",
        "claim_id": 1,
        "error_type": "Numerical Error",
        "potential_cause": (
            "The model may fail to preserve numerical "
            "information associated with scientific entities."
        ),
        "user_impact": (
            "Readers may obtain incorrect information about "
            "RNA precursor length."
        ),
        "future_mitigation": (
            "Evaluate numerical consistency checks and "
            "number-preserving training examples."
        )
    },
    {
        "sample_id": 55,
        "model": "lora",
        "claim_id": 1,
        "error_type": "Terminology Error",
        "potential_cause": (
            "The model may confuse entities participating "
            "in the same neurotransmitter recycling pathway."
        ),
        "user_impact": (
            "Readers may confuse a transporter protein "
            "with a neurotransmitter."
        ),
        "future_mitigation": (
            "Evaluate biomedical domain adaptation and "
            "entity-role consistency checks."
        )
    },
    {
        "sample_id": 62,
        "model": "lora",
        "claim_id": 4,
        "error_type": "Overgeneralization",
        "potential_cause": (
            "The model may omit experimental qualifiers "
            "when compressing scientific findings."
        ),
        "user_impact": (
            "Readers may incorrectly generalize the result "
            "to all Ctnnb1 mutants."
        ),
        "future_mitigation": (
            "Evaluate experimental condition preservation "
            "and improved source selection."
        )
    },
    {
        "sample_id": 94,
        "model": "lora",
        "claim_id": 1,
        "error_type": "Entity Confusion",
        "potential_cause": (
            "The model may incorrectly assign the function "
            "of MyoD to Tristetraprolin."
        ),
        "user_impact": (
            "Readers may misunderstand the molecular "
            "regulation of muscle satellite cell activation."
        ),
        "future_mitigation": (
            "Evaluate entity-relation grounding and "
            "larger biomedical summarization models."
        )
    }
]

error_cases_df = pd.DataFrame(error_cases)

final_error_analysis_df = error_cases_df.merge(
    final_factuality_df[[
        "sample_id",
        "article_id",
        "model",
        "config",
        "claim_id",
        "claim",
        "input_grounded",
        "full_article",
        "input_evidence",
        "full_article_evidence"
    ]],
    on=["sample_id", "model", "claim_id"],
    how="left",
    validate="one_to_one"
)

final_error_analysis_df = final_error_analysis_df[[
    "sample_id",
    "article_id",
    "model",
    "config",
    "claim_id",
    "error_type",
    "claim",
    "input_grounded",
    "full_article",
    "input_evidence",
    "full_article_evidence",
    "potential_cause",
    "user_impact",
    "future_mitigation"
]]

assert len(final_error_analysis_df) == 6
assert final_error_analysis_df["claim"].notna().all()

assert final_error_analysis_df["full_article"].isin([
    "Contradicted",
    "Not Verifiable"
]).all()

display(final_error_analysis_df)

final_error_analysis_df.to_csv(
    reports_dir / "final_error_analysis.csv",
    index=False
)

print("Analyzed Errors:", len(final_error_analysis_df))
print("Unique Error Types:", final_error_analysis_df["error_type"].nunique())
print("Final Error Analysis: Saved Successfully")

,sample_id,article_id,model,config,claim_id,error_type,claim,input_grounded,full_article,input_evidence,full_article_evidence,potential_cause,user_impact,future_mitigation
0,18,d07baa11f86a0b36b6422847bd20bc8b5c20b02150fa75...,lora,C,5,Unsupported Treatment Claim,The described structural changes enable polygl...,Not Verifiable,Not Verifiable,The input describes disease causation and stru...,No treatment efficacy established in the full ...,The model may associate disease-related struct...,Readers may incorrectly assume that a treatmen...,Evaluate source-grounded training and claim-le...
1,42,2d35fc30152d01481c726b2d3f9c26523bb0b8d90112fb...,zero_shot,C,1,Causality Error,Adult LepR reactivation causes irreversible ob...,Contradicted,Contradicted,LepR deficiency in early life led to irreversi...,Early-life deficiency causes persistent metabo...,The model may fail to distinguish early-life r...,Readers may misunderstand the biological cause...,Evaluate causal relation preservation and temp...
2,50,a1429cc8dedc90d48fa9a2294e9586a168d7cd01f0640d...,lora,C,1,Numerical Error,Pol IV- and RDR2-dependent RNAs consist of a s...,Contradicted,Contradicted,P4R2 RNAs are mostly 26–45 nt.,P4R2 RNAs are mostly 26–45 nucleotides long.,The model may fail to preserve numerical infor...,Readers may obtain incorrect information about...,Evaluate numerical consistency checks and numb...
3,55,1339cf0806aac58c617d2e41985ba220acdbd4a680cf8f...,lora,C,1,Terminology Error,The beta-alanine transporter is a neurotransmi...,Contradicted,Contradicted,BalaT is a beta-alanine transporter; histamine...,BalaT is a plasma membrane protein sufficient ...,The model may confuse entities participating i...,Readers may confuse a transporter protein with...,Evaluate biomedical domain adaptation and enti...
4,62,71c1fef70e84f8b2a4bc05c9de4e1620a1503847839c4e...,lora,C,4,Overgeneralization,Ctnnb1 mutants are unable to rescue presynapti...,Not Verifiable,Contradicted,Only the mutant without the transactivation do...,Cell-adhesion-impaired Ctnnb1 rescued deficits...,The model may omit experimental qualifiers whe...,Readers may incorrectly generalize the result ...,Evaluate experimental condition preservation a...
5,94,b91ae9fe362736529a9479c5a2ce277226e3f2df1f535a...,lora,C,1,Entity Confusion,Tristetraprolin is a transcriptional regulator...,Contradicted,Contradicted,"Tristetraprolin promotes decay of MyoD mRNA, w...",MyoD is the myogenic regulatory transcription ...,The model may incorrectly assign the function ...,Readers may misunderstand the molecular regula...,Evaluate entity-relation grounding and larger ...


Analyzed Errors: 6
Unique Error Types: 6
Final Error Analysis: Saved Successfully
